In [1]:
import json
import os
import pandas as pd 
import numpy as np 
from tqdm import tqdm
import torch
from torch import nn, optim
import torch.nn.functional as F
from torchvision import models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn import metrics
from matplotlib import pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier
import seaborn as sns
import cv2
from PIL import Image
import re
from collections import Counter
import nltk
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import ast

In [2]:
# Extract and process captions

ROOT = r"C:\Users\Keelan24\Documents\My Projects\MSC Final"
JSON_PATHS = os.path.join(ROOT,"Capdata","All the annotation","All the annotation")
IMAGE_FOLDERS =  os.path.join(ROOT,"Capdata","UCF crime Extrated frame Dataset","UCF crime Extrated frame Dataset")
os.listdir(JSON_PATHS)

['Abuse.json',
 'Arson.json',
 'Assault.json',
 'Burglary.json',
 'Explosion.json',
 'Fighting.json',
 'Normal.json',
 'Normal_Videos part1.json',
 'Normal_Videos part2.json',
 'Normal_Videos part3.json',
 'Normal_Videos part4.json',
 'Robbery.json',
 'Shooting.json',
 'Shoplifting.json',
 'Stealing.json',
 'Vandalism.json']

In [3]:
def extract_json(json_path, JSON_PATHS = JSON_PATHS):
    p = os.path.join(JSON_PATHS, json_path)
    with open(p, 'r') as f:
        data = json.load(f)
    df = pd.DataFrame.from_dict(data, orient='index')
    df.index.name = 'video_id'

    return df

df = pd.DataFrame()
for i, json_file in enumerate(os.listdir(JSON_PATHS)):
    df = pd.concat([df,extract_json(json_file)])

df.reset_index(inplace=True)

In [4]:
expanded_rows = []

for _, row in df.iterrows():
    video_id = row["video_id"]
    for (start, end), sentence in zip(row["timestamps"], row["sentences"]):
        expanded_rows.append({
            "video_id": video_id,
            "start_time": start,
            "end_time": end,
            "caption": sentence
        })
expanded_df = pd.DataFrame(expanded_rows)

captions_df = expanded_df

captions_df

,video_id,start_time,end_time,caption
0,Abuse001_x264,0.0,5.3,"A woman with short hair, slightly fat, wearing..."
1,Abuse001_x264,7.0,8.5,A man wearing a white shirt and black pants en...
2,Abuse001_x264,7.2,8.5,A man wearing a black shirt and black pants en...
3,Abuse001_x264,8.2,8.9,A man wearing a white shirt and black pants ap...
4,Abuse001_x264,8.9,11.2,A man in black clothes approached a short-hair...
...,...,...,...,...
10996,Vandalism049_x264,243.2,253.4,The bald man held the door with one hand and m...
10997,Vandalism049_x264,253.4,266.5,The bald man held the door with one hand and m...
10998,Vandalism050_x264,0.0,8.0,There are three people next to a black car and...
10999,Vandalism050_x264,8.0,18.5,A man in gray clothes with a hat poured someth...


In [28]:

# Tokenize all captions into words
all_tokens = [word.lower() for cap in expanded_df["caption"] for word in cap.split()]

# Add special tokens
specials = ["<pad>", "<unk>", "<sos>", "<eos>"]
word_counts = Counter(all_tokens)
vocab = {token: idx for idx, token in enumerate(specials + list(word_counts.keys()))}
inv_vocab = {idx: token for token, idx in vocab.items()}

inv_vocab.get(3)

'<eos>'

In [6]:
# Frames

EFFECTIVE_FPS = 3
CLIP_LEN = 32
FRAME_HEIGHT, FRAME_WIDTH = 224, 224
MAX_LENGTH = 50
transform = T.Compose([
        T.Resize((FRAME_HEIGHT, FRAME_WIDTH)), 
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Same transform as used in classification
    ])



In [7]:
records = []

for class_name in os.listdir(IMAGE_FOLDERS):
    class_path = os.path.join(IMAGE_FOLDERS, class_name)
    if not os.path.isdir(class_path):
        continue

    for file_name in os.listdir(class_path):
        if not file_name.endswith(".png"):
            continue

        # Example: Fighting002_x264_frame_390.png
        parts = file_name.split("_frame_")
        video_id = parts[0]
        frame_idx = int(parts[1].split(".")[0])

        # Build a mapping of video_id -> all frame indices
        # We'll group frames belonging to the same video
        records.append({
            "class_name": class_name,
            "video_id": video_id,
            "frame_idx": frame_idx
        })

# Turn into DataFrame
frame_df = pd.DataFrame(records)

In [8]:
clip_records = []

for (class_name, video_id), group in tqdm(frame_df.groupby(["class_name", "video_id"])):
    frame_indices = sorted(group["frame_idx"].tolist())

    # Slide window over frames
    for i in range(0, len(frame_indices) - CLIP_LEN + 1, CLIP_LEN):
        clip_frames = frame_indices[i:i+CLIP_LEN]

        start_frame = clip_frames[0]
        end_frame = clip_frames[-1]

        start_time = start_frame / EFFECTIVE_FPS
        end_time = end_frame / EFFECTIVE_FPS

        clip_records.append({
            "class_name": class_name,
            "video_id": video_id,
            "start_frame": start_frame,
            "end_frame": end_frame,
            "start_time": start_time,
            "end_time": end_time,
            "clip_frames": clip_frames  # (optional: list of all frames)
        })
clip_df = pd.DataFrame(clip_records)
clip_df 

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 531/531 [00:00<00:00, 5111.41it/s]


,class_name,video_id,start_frame,end_frame,start_time,end_time,clip_frames
0,Abuse,Abuse012_x264,70,950,23.333333,316.666667,"[70, 80, 90, 670, 680, 690, 700, 710, 720, 730..."
1,Abuse,Abuse013_x264,0,340,0.000000,113.333333,"[0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 1..."
2,Abuse,Abuse013_x264,350,760,116.666667,253.333333,"[350, 360, 370, 380, 390, 400, 410, 420, 430, ..."
3,Abuse,Abuse013_x264,770,1240,256.666667,413.333333,"[770, 780, 790, 830, 840, 850, 860, 870, 880, ..."
4,Abuse,Abuse013_x264,1280,1980,426.666667,660.000000,"[1280, 1290, 1300, 1310, 1320, 1330, 1340, 135..."
...,...,...,...,...,...,...,...
4855,Vandalism,Vandalism049_x264,6920,7230,2306.666667,2410.000000,"[6920, 6930, 6940, 6950, 6960, 6970, 6980, 699..."
4856,Vandalism,Vandalism049_x264,7240,7550,2413.333333,2516.666667,"[7240, 7250, 7260, 7270, 7280, 7290, 7300, 731..."
4857,Vandalism,Vandalism049_x264,7560,7870,2520.000000,2623.333333,"[7560, 7570, 7580, 7590, 7600, 7610, 7620, 763..."
4858,Vandalism,Vandalism050_x264,0,310,0.000000,103.333333,"[0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 1..."


In [9]:
matched_clips = []

for idx, clip_row in tqdm(clip_df.iterrows()):
    clip_vid = clip_row['video_id']
    clip_start = clip_row['start_time']
    clip_end = clip_row['end_time']

    # Find captions for the same video where there is ANY overlap
    mask = (captions_df['video_id'] == clip_vid) & \
           (captions_df['start_time'] < clip_end) & \
           (captions_df['end_time'] > clip_start)

    matching_captions = captions_df[mask]['caption'].tolist()

    matched_clips.append({
        "class_name": clip_row['class_name'],
        "video_id": clip_vid,
        "start_frame": clip_row['start_frame'],
        "end_frame": clip_row['end_frame'],
        "start_time": clip_start,
        "end_time": clip_end,
        "clip_frames": clip_row['clip_frames'],
        "captions": matching_captions  # Overlapping captions
    })

4860it [00:07, 640.01it/s]


In [10]:
df = pd.DataFrame(matched_clips)

df = df[df['captions'].map(lambda x: len(x) > 0)]
df = df.reset_index(drop=True)
df = df.explode('captions').reset_index(drop=True)
df

,class_name,video_id,start_frame,end_frame,start_time,end_time,clip_frames,captions
0,Abuse,Abuse012_x264,70,950,23.333333,316.666667,"[70, 80, 90, 670, 680, 690, 700, 710, 720, 730...",The woman stopped beating the baby after it st...
1,Abuse,Abuse012_x264,70,950,23.333333,316.666667,"[70, 80, 90, 670, 680, 690, 700, 710, 720, 730...",The woman began to beat the baby. After hittin...
2,Abuse,Abuse012_x264,70,950,23.333333,316.666667,"[70, 80, 90, 670, 680, 690, 700, 710, 720, 730...",The woman grabbed the baby's hand and picked u...
3,Abuse,Abuse012_x264,70,950,23.333333,316.666667,"[70, 80, 90, 670, 680, 690, 700, 710, 720, 730...","After the woman pulled the baby upright, she p..."
4,Abuse,Abuse012_x264,70,950,23.333333,316.666667,"[70, 80, 90, 670, 680, 690, 700, 710, 720, 730...",The woman turned the baby over on her right ar...
...,...,...,...,...,...,...,...,...
4176,Vandalism,Vandalism049_x264,640,950,213.333333,316.666667,"[640, 650, 660, 670, 680, 690, 700, 710, 720, ...",The bald man held the door with one hand and m...
4177,Vandalism,Vandalism049_x264,640,950,213.333333,316.666667,"[640, 650, 660, 670, 680, 690, 700, 710, 720, ...",The bald man held the door with one hand and m...
4178,Vandalism,Vandalism050_x264,0,310,0.000000,103.333333,"[0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 1...",There are three people next to a black car and...
4179,Vandalism,Vandalism050_x264,0,310,0.000000,103.333333,"[0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 1...",A man in gray clothes with a hat poured someth...


In [11]:
nltk.download('punkt_tab')
def tokenize(text):
    return word_tokenize(text.lower())

all_tokens = [token for cap in expanded_df["caption"] for token in tokenize(cap)]

specials = ["<pad>", "<unk>", "<sos>", "<eos>"]
word_counts = Counter(all_tokens)

filtered_tokens = [word for word, freq in word_counts.items() if freq > 2]

vocab = {token: idx for idx, token in enumerate(specials + filtered_tokens)}
inv_vocab = {idx: token for token, idx in vocab.items()}


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Keelan24\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [12]:
def simple_tokenize(text):
    return re.findall(r'\b\w+\b', text.lower())

# Tokenize every caption (each row now has just one caption)
df['tokens'] = df['captions'].apply(lambda x: simple_tokenize(x))

def tokens_to_ids(tokens, vocab):
    sos = vocab["<sos>"]
    eos = vocab["<eos>"]
    unk = vocab["<unk>"]

    ids = [sos] + [vocab.get(token, unk) for token in tokens] + [eos]
    return ids

# Convert tokens to token_ids
df['token_ids'] = df['tokens'].apply(lambda tokens: tokens_to_ids(tokens, vocab))

df


,class_name,video_id,start_frame,end_frame,start_time,end_time,clip_frames,captions,tokens,token_ids
0,Abuse,Abuse012_x264,70,950,23.333333,316.666667,"[70, 80, 90, 670, 680, 690, 700, 710, 720, 730...",The woman stopped beating the baby after it st...,"[the, woman, stopped, beating, the, baby, afte...","[2, 22, 5, 257, 317, 22, 539, 127, 29, 257, 13..."
1,Abuse,Abuse012_x264,70,950,23.333333,316.666667,"[70, 80, 90, 670, 680, 690, 700, 710, 720, 730...",The woman began to beat the baby. After hittin...,"[the, woman, began, to, beat, the, baby, after...","[2, 22, 5, 543, 30, 311, 22, 539, 127, 232, 22..."
2,Abuse,Abuse012_x264,70,950,23.333333,316.666667,"[70, 80, 90, 670, 680, 690, 700, 710, 720, 730...",The woman grabbed the baby's hand and picked u...,"[the, woman, grabbed, the, baby, s, hand, and,...","[2, 22, 5, 316, 22, 539, 345, 71, 15, 24, 25, ..."
3,Abuse,Abuse012_x264,70,950,23.333333,316.666667,"[70, 80, 90, 670, 680, 690, 700, 710, 720, 730...","After the woman pulled the baby upright, she p...","[after, the, woman, pulled, the, baby, upright...","[2, 127, 22, 5, 49, 22, 539, 549, 72, 550, 22,..."
4,Abuse,Abuse012_x264,70,950,23.333333,316.666667,"[70, 80, 90, 670, 680, 690, 700, 710, 720, 730...",The woman turned the baby over on her right ar...,"[the, woman, turned, the, baby, over, on, her,...","[2, 22, 5, 59, 22, 539, 176, 82, 70, 64, 520, ..."
...,...,...,...,...,...,...,...,...,...,...
4176,Vandalism,Vandalism049_x264,640,950,213.333333,316.666667,"[640, 650, 660, 670, 680, 690, 700, 710, 720, ...",The bald man held the door with one hand and m...,"[the, bald, man, held, the, door, with, one, h...","[2, 22, 265, 32, 93, 22, 160, 6, 202, 71, 15, ..."
4177,Vandalism,Vandalism049_x264,640,950,213.333333,316.666667,"[640, 650, 660, 670, 680, 690, 700, 710, 720, ...",The bald man held the door with one hand and m...,"[the, bald, man, held, the, door, with, one, h...","[2, 22, 265, 32, 93, 22, 160, 6, 202, 71, 15, ..."
4178,Vandalism,Vandalism050_x264,0,310,0.000000,103.333333,"[0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 1...",There are three people next to a black car and...,"[there, are, three, people, next, to, a, black...","[2, 78, 115, 80, 428, 380, 30, 4, 16, 122, 15,..."
4179,Vandalism,Vandalism050_x264,0,310,0.000000,103.333333,"[0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 1...",A man in gray clothes with a hat poured someth...,"[a, man, in, gray, clothes, with, a, hat, pour...","[2, 4, 32, 19, 191, 62, 6, 4, 184, 921, 291, 8..."


In [13]:
class VideoCaptionDataset(Dataset):
    def __init__(self, caption_data, video_dir, max_seq_len=64, pad_id=0):
        self.data = caption_data
        self.video_dir = video_dir
        self.max_seq_len = max_seq_len
        self.pad_id = pad_id
        
        # Fix data types (because clip_frames, tokens, token_ids are saved as strings)
        self.data['clip_frames'] = self.data['clip_frames']#.apply(ast.literal_eval)
        self.data['token_ids'] = self.data['token_ids']#.apply(ast.literal_eval)

        # Image transform for frame preprocessing
        self.transform = T.Compose([
            T.Resize((224, 224)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet mean/std
        ])

    def __len__(self):
        return len(self.data)

    def pad_sequence(self, ids):
        if len(ids) >= self.max_seq_len:
            return ids[:self.max_seq_len]
        else:
            return ids + [self.pad_id] * (self.max_seq_len - len(ids))

    def load_frames(self, video_id, frame_indices):
        frames = []
        for idx in frame_indices:
            folder = re.match(r"([A-Za-z]+)\d+_x264", video_id).group(1)
            frame_path = os.path.join(self.video_dir,folder , f"{video_id}_frame_{idx}.png")  # Assuming images are stored as jpg files
            frame = Image.open(frame_path).convert('RGB')
            frame = self.transform(frame)
            frames.append(frame)
        return torch.stack(frames)

    def extract_slowfast_features(self, frames, slow_sample_rate=4):
        # Fast pathway: use all frames
        fast_pathway = frames
        
        # Slow pathway: sample frames with slow_sample_rate
        slow_pathway = frames[::slow_sample_rate]  # Take every 'slow_sample_rate' frame
    
        
        return slow_pathway, fast_pathway

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        # === Load SlowFast features ===
        video_id = row['video_id']
        frame_indices = row['clip_frames']
        
        frames = self.load_frames(video_id, frame_indices)
        
        # Assuming SlowFast model gives us slow and fast pathways
        slow_pathway, fast_pathway = self.extract_slowfast_features(frames)

        # === Caption ===
        token_ids = self.pad_sequence(row['token_ids'])
        token_ids = torch.tensor(token_ids, dtype=torch.long)

        return (slow_pathway, fast_pathway), token_ids


In [14]:
dataset = VideoCaptionDataset(
    df, 
    IMAGE_FOLDERS, 
    max_seq_len=64)

video_features, token_ids = dataset[420]


train_indices, test_indices = train_test_split(range(len(dataset)), test_size=0.2, random_state=42)

# Create subsets for training and testing
train_subset = Subset(dataset, train_indices)
test_subset = Subset(dataset, test_indices)
num_workers = 0
# Create DataLoaders
batch_size = 1
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

# Check one batch from the train_loader
for video_features, token_ids in train_loader:
    print("Video Features (Slow Pathway) Shape:", video_features[0].shape)
    print("Video Features (Fast Pathway) Shape:", video_features[1].shape)
    print("Token IDs Shape:", token_ids.shape)
    break  # Just print the first batch and stop

# Optional: Verify the length of the dataset and the dataloaders
print(f"Length of full dataset: {len(dataset)}")
print(f"Length of train loader: {len(train_loader)}")
print(f"Length of test loader: {len(test_loader)}")






# An example to test outputs work
print(video_features[0].shape)  # slow pathway
print(video_features[1].shape)  # fast pathway
print(token_ids)
print(len(dataset))

Video Features (Slow Pathway) Shape: torch.Size([1, 8, 3, 224, 224])
Video Features (Fast Pathway) Shape: torch.Size([1, 32, 3, 224, 224])
Token IDs Shape: torch.Size([1, 64])
Length of full dataset: 4181
Length of train loader: 3344
Length of test loader: 837
torch.Size([1, 8, 3, 224, 224])
torch.Size([1, 32, 3, 224, 224])
tensor([[  2,  22,  32,  19,  16,  36,  63, 355,   4, 371, 373, 144, 738,  73,
          22, 533,  21,  22, 963, 964,  15,  58,  60, 502,  22, 490,   3,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0]])
4181


In [17]:
class SentryNet(nn.Module): 
    def __init__(self, lstm_hidden_size, vocab_size, embedding_dim, num_layers = 1):
        super(SentryNet, self).__init__()
        # SlowFast encoder
        self.sf = torch.hub.load('facebookresearch/pytorchvideo', 'slowfast_r50', pretrained=True)
        self.sf.blocks[-1] = nn.Identity()

        # Feature encoder LSTM
        self.encoder_lstm = nn.LSTM(input_size=2304, hidden_size=lstm_hidden_size, num_layers=num_layers, batch_first=True)

        # Caption decoder
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.decoder_lstm = nn.LSTM(input_size=embedding_dim, hidden_size=lstm_hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(lstm_hidden_size, vocab_size)

    def forward(self, x, captions=None, teacher_forcing_ratio=0.5):
        batch_size = x[0].size(0)  # Assuming x is a tuple of slow and fast features
        
        # SlowFast feature extraction (separate for slow and fast pathways)
        
        slow_features = x[0] # Slow pathway
        slow_features = slow_features.permute(0, 2, 1, 3, 4)  
        fast_features = x[1].permute(0, 2, 1, 3, 4)   # Fast pathway
    
        # Forward pass through SlowFast model
        features = self.sf([slow_features, fast_features])  # (batch_size, 2304)
        features = features.flatten(1) 
        features = features.unsqueeze(1)  # (batch_size, 1, 2304)
    
        # Encoder LSTM (creates initial hidden states)
        _, (h_n, c_n) = self.encoder_lstm(features)
    
        # Decoder LSTM (generates captions)
        outputs = []
        device = x[0].device  # Get device from slow_features
        input_token = torch.full((batch_size,), vocab["<sos>"], dtype=torch.long, device=device)

        seq_len = captions.size(1) if captions is not None else 20  # max length at inference if no ground truth
        for t in range(seq_len):
            embedded = self.embedding(input_token).unsqueeze(1)  # (batch_size, 1, embedding_dim)
            output, (h_n, c_n) = self.decoder_lstm(embedded, (h_n, c_n))  # (batch_size, 1, hidden_size)
            output = self.fc(output.squeeze(1))  # (batch_size, vocab_size)
            outputs.append(output)
    
            # Decide next input token
            if captions is not None and torch.rand(1).item() < teacher_forcing_ratio:
                input_token = captions[:, t]  # use ground truth
            else:
                input_token = output.argmax(1)  # use model prediction
    
        outputs = torch.stack(outputs, dim=1)  # (batch_size, seq_len, vocab_size)
        return outputs


lstm_hidden_size = 512
vocab_size = len(vocab)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
model = SentryNet(lstm_hidden_size = lstm_hidden_size, num_layers = 1, vocab_size = len(vocab), embedding_dim = 256).to(device)

Using device: cuda


Using cache found in C:\Users\Keelan24/.cache\torch\hub\facebookresearch_pytorchvideo_main


In [18]:
# Get first batch from train_loader
video_features, token_ids = next(iter(train_loader))
token_ids = token_ids.to(device)
video_features = (video_features[0].to(device), video_features[1].to(device))
print(video_features[1].shape)  # Should be [batch_size, channels, frames, height, width]
print(token_ids.shape)

# Pass through the models
output = model(video_features, token_ids)


torch.Size([1, 32, 3, 224, 224])
torch.Size([1, 64])
After SF: torch.Size([1, 2304, 1, 1, 1])


In [29]:
# Setup
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ignore padding token
smooth_fn = SmoothingFunction().method1

ignore_indices = [0, 2]  



num_epochs = 10
losses = np.zeros((2, num_epochs))
bleu_scores = np.zeros(num_epochs)
best_loss = float('inf')

for epoch in range(num_epochs):
    model.train()  
    total_loss = 0

    for video_features, token_ids in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):
        video_features = [feature.to(device) for feature in video_features]
        token_ids = token_ids.to(device)

        optimizer.zero_grad()

        output = model(video_features, token_ids)

        # Reshape outputs
        output = output.view(-1, vocab_size)

        mask = token_ids != 0  # Exclude <pad> (0)
        mask = mask & (token_ids != 2)  # Exclude <sos> (2)
        output = output[mask.view(-1)]
        token_ids_flat = token_ids.view(-1)
        token_ids_flat = token_ids_flat[mask.view(-1)]

        loss = criterion(output, token_ids_flat)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    losses[0, epoch] = avg_train_loss

    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for video_features, token_ids in tqdm(test_loader, desc=f"Validation Epoch {epoch+1}"):
            video_features = [feature.to(device) for feature in video_features]
            token_ids = token_ids.to(device)

            output = model(video_features, token_ids)

            output = output.view(-1, vocab_size)

            mask = token_ids != 0  # Exclude <pad> (0)
            mask = mask & (token_ids != 2)  # Exclude <sos> (2)
            output = output[mask.view(-1)]
            token_ids_flat = token_ids.view(-1)
            token_ids_flat = token_ids_flat[mask.view(-1)]

            loss = criterion(output, token_ids_flat)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(test_loader)
    losses[1, epoch] = avg_val_loss

    # Save best model
    if avg_val_loss < best_loss:
        best_loss = avg_val_loss
        print(f"Saving best model at epoch {epoch + 1}")
        torch.save(model.state_dict(), "/mnt/scratch/od22kob/MSC Final/Sentry-AI/Python/Models/Best_Models/SentryNet_best.pt")

    references = []
    hypotheses = []

    with torch.no_grad():
        for video_features, token_ids in tqdm(test_loader, desc=f"Validation Epoch {epoch+1}"):
            video_features = [feature.to(device) for feature in video_features]
            token_ids = token_ids.to(device)


            output = model(video_features, token_ids, teacher_forcing_ratio=0)  # Important: set teacher forcing to 0

            output_ids = output.argmax(dim=-1)  # Shape: (batch_size, seq_len)

            for ref_tokens, pred_tokens in zip(token_ids, output_ids):
                ref_sentence = [vocab.itos[idx.item()] for idx in ref_tokens if idx.item() not in {0, 1, 2}]
                pred_sentence = [vocab.itos[idx.item()] for idx in pred_tokens if idx.item() not in {0, 1, 2}]
                # 0: <pad>, 1: <sos>, 2: <eos> (example)

                references.append([ref_sentence])  # BLEU expects list of references
                hypotheses.append(pred_sentence)

    # Now calculate BLEU
    bleu_score = 0.0
    for ref, hyp in zip(references, hypotheses):
        bleu_score += sentence_bleu(ref, hyp, smoothing_function=smooth_fn)

    avg_bleu = bleu_score / len(references)
    bleu_scores[epoch] = avg_bleu  # Save it

    print(f"Validation BLEU Score: {avg_bleu:.4f}")
    print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Test BLEU: {avg_bleu:.4f}")


Training Epoch 1:   0%|                                                                                                                                                    | 0/3344 [00:00<?, ?it/s]

After SF: torch.Size([1, 2304, 1, 1, 1])


Training Epoch 1:   0%|                                                                                                                                          | 1/3344 [00:04<3:50:55,  4.14s/it]

After SF: torch.Size([1, 2304, 1, 1, 1])


Training Epoch 1:   0%|                                                                                                                                          | 2/3344 [00:10<5:11:18,  5.59s/it]

...

After SF: torch.Size([1, 2304, 1, 1, 1])


Training Epoch 1:   1%|█▎                                                                                                                                       | 33/3344 [03:28<5:47:40,  6.30s/it]

After SF: torch.Size([1, 2304, 1, 1, 1])


Training Epoch 1:   1%|█▎                                                                                                                                       | 33/3344 [03:34<5:58:12,  6.49s/it]


KeyboardInterrupt: 

In [ ]:
plt.plot(losses[0], label = 'Training')
plt.plot(losses[1], label = 'Testing')
plt.plot(bleu_scores, label = 'Testing')
plt.grid()
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.title("SentryNet (SlowFast+LSTM) Captioning Model Model")
plt.savefig("Training_SentryNet.png")